### 미션 3. 특정 웹소설 회차 목록 크롤링

- 수집 URL: 네이버 웹소설 통합랭킹 내 특정 작품의 상세/회차 목록 페이지
- 수집 목표
  - 회차 제목
  - 회차 링크
  - 평점
- 진행 목표
  - 먼저 1페이지의 회차 목록을 수집합니다.
  - 이후 전체 페이지를 순회하며 회차 목록을 수집합니다.
  - 미션 3의 최종 결과만 CSV 파일로 저장합니다.


In [47]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"
headers = {"User-Agent": user_agent}

BASE_URL = "https://novel.naver.com"
url = "https://novel.naver.com/best/list?novelId=1221743"

page_response = requests.get(url, timeout=5)
first_page_soup = BeautifulSoup(page_response.text, "html.parser")
pagination = [tag["href"] for tag in first_page_soup.select("div.default_paging a")]
page_soups = [first_page_soup]

# 페이지네이션에서 추출한 링크들을 하나씩 돌면서 나머지 페이지들도 요청
for page_href in pagination:
    page_url = BASE_URL + page_href
    page_response = requests.get(page_url, timeout=5)
    # 파싱한 결과를 같은 리스트에 계속해서 추가
    page_soups.append(BeautifulSoup(page_response.text, "html.parser"))

all_records = []

# 모든 페이지의 soup을 한 번에 순회
for page_soup in page_soups:
    episode_container = page_soup.select("li.volumeComment")

    for episode in episode_container:
        title = episode.select_one("div.list_info p.subj").get_text(strip=True)
        link = BASE_URL + episode.select_one("a.list_item")["href"]
        star = (
            episode.select_one("div.list_info span.score_area")
            .get_text(strip=True)
            .replace("별점", "")
        )
        # [제목, 링크, 평점]을 한 행으로 만들어 결과 리스트에 추가
        all_records.append([title, link, star])



print(len(all_records))

38


In [50]:
# dataframe 전환

df_episodes = pd.DataFrame(all_records, columns=["제목", "링크", "평점"])
print(df_episodes.head())

            제목                                                 링크    평점
0  38화. 결사단(2)  https://novel.naver.com/best/detail?novelId=12...  10.0
1     37화. 결사단  https://novel.naver.com/best/detail?novelId=12...  10.0
2   36화. 막간(5)  https://novel.naver.com/best/detail?novelId=12...  10.0
3   35화. 막간(4)  https://novel.naver.com/best/detail?novelId=12...  10.0
4   34화. 막간(3)  https://novel.naver.com/best/detail?novelId=12...  10.0


In [52]:
# csv 파일 저장

df_episodes.to_csv("naver_novel.csv", index=False, encoding="utf-8-sig")
print("저장완료")

저장완료
